# Notebook 3: Topology Stress Test

The paper identifies four canonical motifs — chain, fan-out, diamond, merge —
each with different failure surfaces and governance implications.

This notebook builds each motif, stresses it, and shows how topology
determines where governance has leverage.

**Motifs tested:**
1. Single delegation (baseline)
2. Linear chain (depth 1–6)
3. Fan-out (degree 1–5)
4. Diamond pattern (conditional fragility)
5. Full SDLC pipeline (combined motifs)

In [ ]:
from minimal_oversight import analyze_pipeline
from minimal_oversight.models import Node, PipelineGraph, AggregationType
from minimal_oversight.capacity import compute_pipeline_capacity, compute_c_op, check_feasibility
from minimal_oversight.topology import detect_motifs, rank_nodes_by_risk, conditional_fragility, delegation_centrality
from minimal_oversight.intervention import explain_failure_surface
from minimal_oversight.simulation import simulate_pipeline, SimulationConfig
from minimal_oversight import _formulae as F

import numpy as np

## Helper: build any motif

In [ ]:
def make_node(name, sigma_skill=0.55, catch_rate=0.65, review_cap=0.50, **kw):
    return Node(name, sigma_skill=sigma_skill, catch_rate=catch_rate,
                review_capacity=review_cap, **kw)

def make_chain(depth, **kw):
    """Linear chain of `depth` identical nodes."""
    nodes = [make_node(f"L{i}", **kw) for i in range(depth)]
    p = PipelineGraph(nodes)
    for i in range(depth - 1):
        p.add_edge(f"L{i}", f"L{i+1}")
    return p

def make_fanout(degree, **kw):
    """One source fanning out to `degree` children, merging at a gate."""
    source = make_node("source", **kw)
    children = [make_node(f"branch_{i}", **kw) for i in range(degree)]
    gate = make_node("gate", aggregation=AggregationType.PRODUCT, **kw)
    p = PipelineGraph([source] + children + [gate])
    for c in children:
        p.add_edge("source", c.name)
        p.add_edge(c.name, "gate")
    return p

def make_diamond(**kw):
    """A → {B, C} → D. Classic diamond with shared upstream."""
    a = make_node("A", **kw)
    b = make_node("B", **kw)
    c = make_node("C", **kw)
    d = make_node("D", aggregation=AggregationType.PRODUCT, **kw)
    p = PipelineGraph([a, b, c, d])
    p.add_edge("A", "B")
    p.add_edge("A", "C")
    p.add_edge("B", "D")
    p.add_edge("C", "D")
    return p

def make_single(**kw):
    return PipelineGraph([make_node("agent", **kw)])

## 1. Motif comparison: capacity and masking

Same agent parameters, different topologies. How does structure change the outcome?

In [ ]:
params = dict(sigma_skill=0.55, catch_rate=0.65, review_cap=0.50)

topologies = [
    ("Single node",       make_single(**params)),
    ("Chain D=2",         make_chain(2, **params)),
    ("Chain D=3",         make_chain(3, **params)),
    ("Chain D=5",         make_chain(5, **params)),
    ("Fan-out k=2",       make_fanout(2, **params)),
    ("Fan-out k=3",       make_fanout(3, **params)),
    ("Fan-out k=5",       make_fanout(5, **params)),
    ("Diamond",           make_diamond(**params)),
]

print(f"{'Topology':<20} {'Nodes':>6} {'Depth':>6} {'C_op':>8} {'Feasible@0.50':>14} {'Feasible@0.70':>14}")
print("-" * 72)

for name, p in topologies:
    c = compute_c_op(p)
    f50 = check_feasibility(p, p_min=0.50)
    f70 = check_feasibility(p, p_min=0.70)
    n_nodes = len(p.nodes)
    depth = p.depth
    s50 = "YES" if f50.feasible else "NO"
    s70 = "YES" if f70.feasible else "NO"
    print(f"{name:<20} {n_nodes:>6} {depth:>6} {c:>8.3f} {s50:>14} {s70:>14}")

print("\nDepth kills capacity. Fan-out preserves capacity but amplifies cascade risk.")

## 2. Chain stress test: depth vs capacity

The paper's recursive formula (Eq. 11) predicts that quality decreases
monotonically with depth. Here we verify it for multiple catch rates.

In [ ]:
print(f"{'Depth':<8}", end="")
catch_rates = [0.30, 0.50, 0.65, 0.80, 0.90]
for c in catch_rates:
    print(f"{'c='+str(c):>10}", end="")
print()
print("-" * (8 + 10 * len(catch_rates)))

for depth in range(1, 8):
    print(f"{depth:<8}", end="")
    for c in catch_rates:
        # Theory: recursive chain quality
        q = F.recursive_chain_quality(depth, 0.55, c, 10, 2)
        print(f"{q:>10.3f}", end="")
    print()

print(f"\n{'p_min':<8}", end="")
for _ in catch_rates:
    print(f"{'0.500':>10}", end="")
print()

print("\nStrong correctors (c=0.90) maintain quality above p_min=0.50 even at D=7.")
print("Weak correctors (c=0.30) breach p_min by D=5.")

## 3. Fan-out stress test: cascade amplification

When the source node fails, all branches receive degraded input.
The question: how much does fan-out degree amplify the cascade?

In [ ]:
print("Fan-out cascade: removing the source corrector\n")
print(f"{'Fan-out k':<12} {'C_op (with)':>12} {'C_op (without)':>15} {'Δ':>8} {'DC(source)':>12}")
print("-" * 62)

for k in [1, 2, 3, 4, 5]:
    # With source corrector
    p_with = make_fanout(k)
    c_with = compute_c_op(p_with)
    dc_source = delegation_centrality(p_with, "source")

    # Without source corrector (catch_rate = 0)
    p_without = make_fanout(k)
    p_without.get_node("source").catch_rate = 0.0
    c_without = compute_c_op(p_without)

    delta = c_without - c_with
    print(f"{k:<12} {c_with:>12.3f} {c_without:>15.3f} {delta:>+8.3f} {dc_source:>12.1f}")

print("\nDelegation centrality and cascade damage both grow with fan-out degree.")
print("This is why the SOTA model should go to the corrector at fan-out nodes.")

## 4. Diamond stress test: conditional fragility

The diamond pattern (A → {B,C} → D) creates correlated errors at D.
Average quality looks fine, but conditional failure is severe.

In [ ]:
diamond = make_diamond()
r = analyze_pipeline(diamond, p_min=0.50)

print("Diamond pipeline analysis:")
print(f"  C_op = {r.feasibility.c_op:.3f}")
print(f"  Feasible @ p_min=0.50: {r.feasibility.feasible}")
print()

# Motifs detected
for m in r.motifs:
    print(f"  [{m.motif.value}] {m.risk_description}")
print()

# Node risks
print(f"{'Node':<8} {'DC':>8} {'M*':>8} {'SOTA':>8}")
print("-" * 36)
for risk in r.node_risks:
    sota = f"{risk.sota_score:.3f}" if risk.sota_score else "  n/a"
    m = f"{risk.masking_index:.2f}" if risk.masking_index else " n/a"
    print(f"{risk.name:<8} {risk.delegation_centrality:>8.1f} {m:>8} {sota:>8}")

### Conditional fragility analysis

What is P(D correct | A correct) vs P(D correct | A error)?

In [ ]:
print("Conditional fragility at D, varying A's corrector catch rate:\n")
print(f"{'c_A':>8} {'Fragility ratio':>16} {'Interpretation'}")
print("-" * 55)

for c_a in [0.0, 0.3, 0.5, 0.65, 0.9]:
    d = make_diamond()
    # Set corrected qualities at B and C based on A's catch rate
    sigma_raw_a = F.sigma_raw_fixed_point(0.55, 10, 2)
    sigma_corr_a = F.sigma_corr_fixed_point(sigma_raw_a, c_a)
    
    # B and C receive A's corrected output
    sigma_skill_eff = 0.55 * sigma_corr_a
    sigma_raw_bc = F.sigma_raw_fixed_point(sigma_skill_eff, 10, 2)
    sigma_corr_bc = F.sigma_corr_fixed_point(sigma_raw_bc, 0.65)
    
    parent_corrs = {"B": sigma_corr_bc, "C": sigma_corr_bc}
    frag = conditional_fragility(d, "D", parent_corrs, shared_source_catch_rate=c_a)
    
    if frag > 1.3:
        interp = "HIGH fragility — fix A upstream"
    elif frag > 1.05:
        interp = "Moderate — A partially decorrelates errors"
    else:
        interp = "Low — A's corrector decorrelates B,C errors"
    
    print(f"{c_a:>8.2f} {frag:>16.2f}× {interp}")

print("\nThe prescription: fix A (upstream) rather than adding redundancy at D.")
print("A's corrector doesn't just improve A — it decorrelates B and C's errors.")

## 5. Aggregation type comparison

The merge node's aggregation function determines how errors combine.
Product is harshest, weakest-link is most honest, mean dilutes everything.

In [ ]:
print(f"{'Aggregation':<15} {'C_op':>8} {'M*(gate)':>10}")
print("-" * 36)

for agg_type, agg_enum in [
    ("product", AggregationType.PRODUCT),
    ("weakest_link", AggregationType.WEAKEST_LINK),
    ("mean", AggregationType.WEIGHTED_MEAN),
]:
    p = make_fanout(3)
    p.get_node("gate").aggregation = agg_enum
    r = analyze_pipeline(p, p_min=0.50)
    gate = p.get_node("gate")
    m = gate.masking_index or 0
    print(f"{agg_type:<15} {r.feasibility.c_op:>8.3f} {m:>10.2f}")

print("\nProduct produces the highest masking — errors compound multiplicatively.")
print("Weakest link is more 'honest' (lower M*) but most fragile.")
print("Mean dilutes both errors and masking.")

## 6. Delegation centrality across topologies

DC(v) measures governance leverage: where does a single correction
propagate the farthest?

In [ ]:
# Build the full SDLC pipeline for comparison
def make_sdlc():
    nodes = [
        make_node("gen"), make_node("rev"),
        make_node("test"), make_node("req"), make_node("sec"),
        make_node("merge", aggregation=AggregationType.PRODUCT),
    ]
    p = PipelineGraph(nodes)
    p.add_edge("gen", "rev")
    p.add_edge("rev", "test")
    p.add_edge("rev", "req")
    p.add_edge("rev", "sec")
    p.add_edge("test", "merge")
    p.add_edge("req", "merge")
    p.add_edge("sec", "merge")
    return p

topologies_dc = [
    ("Chain D=4",    make_chain(4)),
    ("Fan-out k=3",  make_fanout(3)),
    ("Diamond",      make_diamond()),
    ("SDLC",         make_sdlc()),
]

for topo_name, p in topologies_dc:
    print(f"\n--- {topo_name} ---")
    risks = rank_nodes_by_risk(p)
    for risk in risks:
        sota = f"{risk.sota_score:.3f}" if risk.sota_score is not None else "  n/a"
        motif_str = ", ".join(m.value for m in risk.motifs) if risk.motifs else "-"
        print(f"  {risk.name:<12} DC={risk.delegation_centrality:>6.1f}  "
              f"SOTA={sota}  fan_out={risk.fan_out_degree}  motifs=[{motif_str}]")

## 7. Stress test: shared upstream failure

What happens to each topology when the entry node's skill drops?

In [ ]:
print(f"{'σ_skill(entry)':<16}", end="")
topo_names = ["Chain D=3", "Fan-out k=3", "Diamond"]
for t in topo_names:
    print(f"{t:>14}", end="")
print()
print("-" * (16 + 14 * len(topo_names)))

for entry_skill in [0.80, 0.65, 0.55, 0.45, 0.35, 0.25]:
    print(f"{entry_skill:<16.2f}", end="")
    
    # Chain: degrade L0
    chain = make_chain(3)
    chain.get_node("L0").sigma_skill = entry_skill
    c_chain = compute_c_op(chain)
    print(f"{c_chain:>14.3f}", end="")
    
    # Fan-out: degrade source
    fo = make_fanout(3)
    fo.get_node("source").sigma_skill = entry_skill
    c_fo = compute_c_op(fo)
    print(f"{c_fo:>14.3f}", end="")
    
    # Diamond: degrade A
    dia = make_diamond()
    dia.get_node("A").sigma_skill = entry_skill
    c_dia = compute_c_op(dia)
    print(f"{c_dia:>14.3f}", end="")
    
    print()

print("\nAll topologies degrade, but fan-out and diamond degrade faster")
print("because the upstream failure propagates to multiple paths.")

## 8. Motif detection in practice

Build a more complex pipeline and let the motif detector find the patterns.

In [ ]:
# Complex pipeline: two generators, shared reviewer, parallel checks, merge
complex_nodes = [
    make_node("gen_code", sigma_skill=0.60),
    make_node("gen_docs", sigma_skill=0.70),
    make_node("reviewer", sigma_skill=0.55),
    make_node("lint", sigma_skill=0.80),
    make_node("test", sigma_skill=0.55),
    make_node("security", sigma_skill=0.50),
    make_node("merge", sigma_skill=0.55, aggregation=AggregationType.PRODUCT),
]
cp = PipelineGraph(complex_nodes)
cp.add_edge("gen_code", "reviewer")
cp.add_edge("gen_docs", "reviewer")
cp.add_edge("reviewer", "lint")
cp.add_edge("reviewer", "test")
cp.add_edge("reviewer", "security")
cp.add_edge("lint", "merge")
cp.add_edge("test", "merge")
cp.add_edge("security", "merge")

print(f"Complex pipeline: {cp}")
print(f"Depth: {cp.depth}\n")

motifs = detect_motifs(cp)
print("Detected motifs:")
for m in motifs:
    print(f"  [{m.motif.value:<10}] nodes={m.nodes}")
    print(f"               {m.risk_description}")

# Full analysis
print()
r = analyze_pipeline(cp, p_min=0.50)
print(f"C_op = {r.feasibility.c_op:.3f}")
print(f"Feasible: {r.feasibility.feasible}")
print(f"Bottleneck: {r.feasibility.bottleneck_node}")
print(f"\nTop recommendations:")
for rec in r.recommendations[:3]:
    print(f"  {rec.priority}. [{rec.target_node}] {rec.action}")

## 9. Governance gap: how much does better governance help?

The governance-complexity frontier (Paper Demo 9): better governance
(higher K/N, Fisher-prioritized routing) shifts the curve up and right.

In [ ]:
print("Autonomy time vs process entropy for different review capacities:\n")
print(f"{'H(W) bits':>10}", end="")
review_levels = [0.30, 0.50, 0.80]
for kn in review_levels:
    print(f"{'K/N='+str(kn):>12}", end="")
print()
print("-" * (10 + 12 * len(review_levels)))

for h_w in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0]:
    print(f"{h_w:>10.1f}", end="")
    for kn in review_levels:
        # Approximate: higher K/N raises C_op and reduces lambda
        p = make_chain(3)
        for name in p.nodes:
            p.get_node(name).review_capacity = kn
        c_op = compute_c_op(p)
        # Lambda decreases with better governance
        lam = 0.03 - 0.01 * kn  # simplified model
        t = F.autonomy_time(c_op, 0.50, lam, h_w, mu_eff=0.005)
        if t > 0:
            print(f"{t:>12.1f}", end="")
        else:
            print(f"{'---':>12}", end="")
    print()

print("\nBetter governance (higher K/N) shifts the frontier up and right:")
print("  - Higher T*_auto at every complexity level")
print("  - The cliff (T*_auto → 0) moves to higher H(W)")

## 10. Summary: topology as a governance object

| Motif | Failure surface | What grows | Governance action |
|-------|----------------|------------|-------------------|
| Chain | Layer-by-layer attenuation | Masking + quality loss with depth | Improve upstream first |
| Fan-out | Shared upstream failure | One failure → multiple branches | Prioritize high-DC nodes |
| Diamond | Correlated errors | Conditional fragility spikes | Fix shared source, not merge |
| Merge | Bottleneck path | Throughput limited by weakest | Invest where ∂C_op/∂c(v) is largest |

In [ ]:
# Final comparison table
print("\n=== Topology Comparison (σ_skill=0.55, c=0.65, p_min=0.50) ===\n")
print(f"{'Topology':<20} {'C_op':>8} {'B_eff':>8} {'Bottleneck':>12} {'Key risk'}")
print("-" * 75)

for name, p in [
    ("Single", make_single(**params)),
    ("Chain D=3", make_chain(3, **params)),
    ("Fan-out k=3", make_fanout(3, **params)),
    ("Diamond", make_diamond(**params)),
    ("SDLC", make_sdlc()),
]:
    r = analyze_pipeline(p, p_min=0.50)
    bn = r.feasibility.bottleneck_node or "-"
    
    if name == "Single":
        risk = "Baseline masking"
    elif "Chain" in name:
        risk = "Depth-driven quality loss"
    elif "Fan" in name:
        risk = "Cascade amplification"
    elif "Diamond" in name:
        risk = "Conditional fragility"
    else:
        risk = "Combined: cascade + fragility"
    
    print(f"{name:<20} {r.feasibility.c_op:>8.3f} {r.feasibility.b_eff:>8.4f} {bn:>12} {risk}")